
# Phase 1 — USA Fortnight Feature Extraction

Extract 15-day (fortnight) satellite features per sugarcane field for **Texas, Louisiana, and Florida**.

**Output:** `usa_fortnight_features.csv` (downloads in Colab)

Run in Google Colab or any environment with Earth Engine access.


In [ ]:

# --- CONFIG ---
EE_PROJECT = 'ee-junaidali101452'
EE_ASSET = 'projects/ee-junaidali101452/assets/FL_TX_LA_Geometries_Confidence'

# 'export' = batch to Drive (full run) | 'interactive' = one-period preview in notebook
EXTRACT_MODE = 'export'

# Pilot: set MAX_FIELDS=50 for a quick test; None for full extraction
MAX_FIELDS = None

# Optional: 'Texas', 'Louisiana', 'Florida', or None for all states
STATE_FILTER = 'Texas'

# Fortnight extraction years (aligned with existing FYP Texas pipeline)
COLLECTION_YEARS = [
    {'start_date': '2019-01-01', 'end_date': '2019-12-31', 'year': 2019},
    {'start_date': '2021-01-01', 'end_date': '2021-12-31', 'year': 2021},
    {'start_date': '2022-01-01', 'end_date': '2022-12-31', 'year': 2022},
]

EXPORT_FOLDER = 'GEE_Fortnight_Exports'
EXPORT_SCALE = 30
EXPORT_TILE_SCALE = 4
OUTPUT_CSV = 'usa_fortnight_features.csv'
DRIVE_EXPORT_PATH = '/content/drive/MyDrive/GEE_Fortnight_Exports'

# Asset properties from CDL reduceToVectors: label=cultivated(2), mean=cropland/sugarcane(45), area=acres
STATE_BBOX_COORDS = {
    'Florida':   [-87.6348, 24.3963, -79.9743, 31.0009],
    'Texas':     [-106.6456, 25.8371, -93.5083, 36.5007],
    'Louisiana': [-94.0430, 28.9281, -88.7584, 33.0192],
}

# Interactive validation: use a period that previously failed batch export
INTERACTIVE_PREVIEW_START = '2019-01-01'

In [ ]:

import ee
import pandas as pd
from datetime import datetime, timedelta

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

if IN_COLAB:
    drive.mount('/content/drive')

STATE_BBOX = {
    name: ee.Geometry.Rectangle(coords)
    for name, coords in STATE_BBOX_COORDS.items()
}


def _in_bbox(lon, lat, coords):
    return lon.gte(coords[0]).And(lon.lte(coords[2])).And(lat.gte(coords[1])).And(lat.lte(coords[3]))


def assign_state_from_geometry(feature):
    """Match asset creation: FL / LA / TX clip rectangles (FL checked before LA before TX)."""
    centroid = feature.geometry().centroid(100)
    lon = ee.Number(centroid.coordinates().get(0))
    lat = ee.Number(centroid.coordinates().get(1))

    in_florida = _in_bbox(lon, lat, STATE_BBOX_COORDS['Florida'])
    in_louisiana = _in_bbox(lon, lat, STATE_BBOX_COORDS['Louisiana'])
    in_texas = _in_bbox(lon, lat, STATE_BBOX_COORDS['Texas'])

    return ee.Feature(feature).set(
        'state',
        ee.String(
            ee.Algorithms.If(
                in_florida, 'Florida',
                ee.Algorithms.If(
                    in_louisiana, 'Louisiana',
                    ee.Algorithms.If(in_texas, 'Texas', 'Unknown')
                )
            )
        )
    )


fc = ee.FeatureCollection(EE_ASSET)
print('Total sugarcane fields in asset:', fc.size().getInfo())
print('Cultivated (label=2):', fc.aggregate_histogram('label').getInfo())
print('Cropland sugarcane (mean=45):', fc.aggregate_histogram('mean').getInfo())

if STATE_FILTER:
    # Same clip geometry used when exporting TX / LA / FL from CDL
    fc = fc.filterBounds(STATE_BBOX[STATE_FILTER]).map(
        lambda f: f.set('state', STATE_FILTER)
    )
    print(f"Filtered to {STATE_FILTER} (CDL clip bbox):", fc.size().getInfo())
else:
    fc = fc.map(assign_state_from_geometry)
    print('Fields by inferred state:', fc.aggregate_histogram('state').getInfo())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total sugarcane fields in asset: 33241
Cultivated (label=2): {'2': 33241}
Cropland sugarcane (mean=45): {'45.0': 33241}
Filtered to Texas (CDL clip bbox): 706


In [ ]:

# State annual sugarcane yields (tons/acre) — USDA NASS reference for Phase 3 mapping
STATE_ANNUAL_YIELDS = {
    'Florida':   {2017: 41.1, 2018: 41.9, 2019: 43.0, 2021: 42.6, 2022: 44.6},
    'Louisiana': {2017: 32.8, 2018: 35.4, 2019: 28.1, 2021: 29.3, 2022: 32.3},
    'Texas':     {2017: 37.1, 2018: 36.6, 2019: 33.8, 2021: 30.9, 2022: 22.6},
}

state_yield_lookup = {
    (state, year): yield_tpa
    for state, years in STATE_ANNUAL_YIELDS.items()
    for year, yield_tpa in years.items()
}

pd.DataFrame(
    [(s, y, v) for s, years in STATE_ANNUAL_YIELDS.items() for y, v in years.items()],
    columns=['state', 'year', 'state_annual_yield'],
)

# Preview one field — label/mean are CDL band values, not state
sample = fc.first().getInfo()
print('Sample properties:', sample.get('properties'))
print('Sample coords:', sample['geometry']['coordinates'][0][0])

Sample properties: {'area': 26.514191151980178, 'label': 2, 'mean': 45, 'state': 'Texas'}
Sample coords: [-98.02175195359987, 26.251254040603015]


In [ ]:

def generate_fortnight_starts(start_date, end_date):
    """Return fortnight start dates (1st and 15th of each month) within range."""
    dates = []
    current = datetime.strptime(start_date, '%Y-%m-%d')
    end = datetime.strptime(end_date, '%Y-%m-%d')
    while current <= end:
        dates.append(current.strftime('%Y-%m-%d'))
        if current.day == 1:
            current = current.replace(day=15)
        elif current.month == 12:
            break
        else:
            current = (current.replace(day=28) + timedelta(days=4)).replace(day=1)
    return dates

fortnight_periods = []
for entry in COLLECTION_YEARS:
    for start in generate_fortnight_starts(entry['start_date'], entry['end_date']):
        fortnight_periods.append({
            'fortnight_start': start,
            'year': entry['year'],
        })

print(f'Total fortnight windows: {len(fortnight_periods)}')
fortnight_periods[:5], '...', fortnight_periods[-2:]

Total fortnight windows: 72


([{'fortnight_start': '2019-01-01', 'year': 2019},
  {'fortnight_start': '2019-01-15', 'year': 2019},
  {'fortnight_start': '2019-02-01', 'year': 2019},
  {'fortnight_start': '2019-02-15', 'year': 2019},
  {'fortnight_start': '2019-03-01', 'year': 2019}],
 '...',
 [{'fortnight_start': '2022-12-01', 'year': 2022},
  {'fortnight_start': '2022-12-15', 'year': 2022}])

In [ ]:
# Assign stable field_id on server and optionally limit for pilot runs
field_count = fc.size()
field_list = fc.toList(field_count)

fc = ee.FeatureCollection(
    ee.List.sequence(0, field_count.subtract(1)).map(
        lambda i: ee.Feature(field_list.get(i)).set('field_id', ee.Number(i).add(1))
    )
)

if MAX_FIELDS is not None:
    fc = fc.limit(MAX_FIELDS)

if STATE_FILTER:
    EXTRACTION_REGION = STATE_BBOX[STATE_FILTER]
else:
    EXTRACTION_REGION = fc.geometry().bounds()

print('Fields queued for extraction:', fc.size().getInfo())
print('Extraction region:', STATE_FILTER or 'all states (fc bounds)')

Fields queued for extraction: 706
Extraction region: Texas


In [ ]:

# --- Earth Engine collections (static references) ---
NASA_srtm = ee.Image('USGS/SRTMGL1_003')


def mask_clouds(image):
    qa60 = image.select('QA60')
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa60.bitwiseAnd(cloud_bit_mask).eq(0).And(qa60.bitwiseAnd(cirrus_bit_mask).eq(0))
    return image.updateMask(mask).copyProperties(image, ['system:time_start'])


def calculate_indices(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    gndvi = image.normalizedDifference(['B8', 'B3']).rename('GNDVI')
    evi = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {'NIR': image.select('B8'), 'RED': image.select('B4'), 'BLUE': image.select('B2')}
    ).rename('EVI')
    red = image.select('B4')
    nir = image.select('B8')
    savi = nir.subtract(red).divide(nir.add(red).add(0.5)).multiply(1.5).rename('SAVI')
    vci = ndvi.expression('(NDVI - 0) / (0.8 - 0) * 100', {'NDVI': ndvi}).rename('VCI')
    tvi = image.expression('0.5 * (NIR - RED) / (NIR + RED + 0.5)', {'NIR': nir, 'RED': red}).rename('TVI')
    bi = image.expression('sqrt((RED*RED) + (NIR*NIR))', {'RED': red, 'NIR': nir}).rename('BI')
    bi2 = image.expression('(RED + NIR) / 2', {'RED': red, 'NIR': nir}).rename('BI2')
    ci = image.expression('(NIR / RED) - 1', {'NIR': nir, 'RED': red}).rename('CI')
    ci1 = image.expression('(RED / NIR) - 1', {'RED': red, 'NIR': nir}).rename('CI1')
    satvi = image.expression('(NIR - RED - 0.5) / (NIR + RED + 0.5)', {'NIR': nir, 'RED': red}).rename('SATVI')
    hvsi = image.expression(
        'sqrt((NIR - RED)*(NIR - RED) + (NIR - GREEN)*(RED - GREEN))',
        {'NIR': nir, 'RED': red, 'GREEN': image.select('B3')}
    ).rename('HVSI')
    soci = image.expression('(NIR - RED) / (NIR + RED)', {'NIR': nir, 'RED': red}).rename('SOCI')
    asi = image.expression('(NIR - RED) / (NIR + RED + 0.5)', {'NIR': nir, 'RED': red}).rename('ASI')
    bsi = image.expression('((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))', {
        'SWIR': image.select('B11'), 'RED': red, 'NIR': nir, 'BLUE': image.select('B2')
    }).rename('BSI')
    msavi = image.expression(
        '(2 * NIR + 1 - sqrt((2 * NIR + 1)*(2 * NIR + 1) - 8 * (NIR - RED))) / 2',
        {'NIR': nir, 'RED': red}
    ).rename('MSAVI')
    return image.addBands([ndvi, gndvi, evi, savi, vci, tvi, bi, bi2, ci, ci1, satvi, hvsi, soci, asi, bsi, msavi])


def cloud_mask_landsat(image):
    qa = image.select('QA_PIXEL')
    cloud = 1 << 3
    cirrus = 1 << 9
    mask = qa.bitwiseAnd(cloud).eq(0).And(qa.bitwiseAnd(cirrus).eq(0))
    return image.updateMask(mask)


def calculate_indices_landsat(img):
    nbr = img.normalizedDifference(['SR_B5', 'SR_B7']).rename('NBR')
    ndmi = img.normalizedDifference(['SR_B5', 'SR_B6']).rename('NDMI')
    ndwi = img.normalizedDifference(['SR_B3', 'SR_B5']).rename('NDWI')
    ndbi = img.normalizedDifference(['SR_B6', 'SR_B5']).rename('NDBI')
    ndbai = img.normalizedDifference(['SR_B6', 'SR_B7']).rename('NDBaI')
    mndwi = img.normalizedDifference(['SR_B3', 'SR_B6']).rename('MNDWI')
    return img.addBands([nbr, ndmi, ndwi, ndbi, ndbai, mndwi])


S2_BANDS = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'QA60']
S2_INDEX_BANDS = ['NDVI', 'EVI', 'GNDVI', 'SAVI', 'VCI', 'TVI', 'BI', 'BI2', 'CI', 'CI1', 'SATVI', 'HVSI', 'SOCI', 'ASI', 'BSI', 'MSAVI']
LS_BANDS = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'NBR', 'NDMI', 'NDWI', 'NDBI', 'NDBaI', 'MNDWI']

In [ ]:
L8_RENAME = {
    'SR_B1': 'SR8_B1', 'SR_B2': 'SR8_B2', 'SR_B3': 'SR8_B3', 'SR_B4': 'SR8_B4',
    'SR_B5': 'SR8_B5', 'SR_B6': 'SR8_B6', 'SR_B7': 'SR8_B7',
    'NBR': 'NBR_8', 'NDMI': 'NDMI_8', 'NDWI': 'NDWI_8',
    'NDBI': 'NDBI_8', 'NDBaI': 'NDBaI_8', 'MNDWI': 'MNDWI_8',
}
L9_RENAME = {
    'SR_B1': 'SR9_B1', 'SR_B2': 'SR9_B2', 'SR_B3': 'SR9_B3', 'SR_B4': 'SR9_B4',
    'SR_B5': 'SR9_B5', 'SR_B6': 'SR9_B6', 'SR_B7': 'SR9_B7',
    'NBR': 'NBR_9', 'NDMI': 'NDMI_9', 'NDWI': 'NDWI_9',
    'NDBI': 'NDBI_9', 'NDBaI': 'NDBaI_9', 'MNDWI': 'MNDWI_9',
}


def empty_placeholder(output_names):
    """Fully masked multi-band placeholder — reduceRegions returns null when no scenes exist."""
    bands = ee.Image.cat([ee.Image.constant(0).rename([name]) for name in output_names])
    return bands.updateMask(ee.Image(0))


def safe_median(ic, source_bands, output_names):
    median = ic.median().select(source_bands).rename(output_names)
    return ee.Image(ee.Algorithms.If(ic.size().gt(0), median, empty_placeholder(output_names)))


def safe_landsat_median(ic, rename_map):
    processed = ic.map(cloud_mask_landsat).map(calculate_indices_landsat)
    source_bands = list(rename_map.keys())
    output_names = list(rename_map.values())
    return safe_median(processed, source_bands, output_names)


def filter_ic(collection_id, start_date_str, end_date_str, region):
    return (
        ee.ImageCollection(collection_id)
        .filterDate(start_date_str, end_date_str)
        .filterBounds(region)
    )


def build_period_composite(start_date_str, end_date_str, region):
    """Single multi-band image for one fortnight — used by reduceRegions (server-side)."""
    s2_ic = filter_ic('COPERNICUS/S2_SR_HARMONIZED', start_date_str, end_date_str, region)
    l8_ic = filter_ic('LANDSAT/LC08/C02/T1_L2', start_date_str, end_date_str, region)
    l9_ic = filter_ic('LANDSAT/LC09/C02/T1_L2', start_date_str, end_date_str, region)
    modis_ic = filter_ic('MODIS/061/MCD15A3H', start_date_str, end_date_str, region)
    smap_ic = filter_ic('NASA/SMAP/SPL4SMGP/008', start_date_str, end_date_str, region)

    s2_proc = s2_ic.select(S2_BANDS).map(mask_clouds).map(calculate_indices)
    s2_median = safe_median(s2_proc, S2_BANDS + S2_INDEX_BANDS, S2_BANDS + S2_INDEX_BANDS)

    s2_count = ee.Image(
        ee.Algorithms.If(
            s2_proc.size().gt(0),
            s2_proc.select('B4').count().rename('s2_scene_count'),
            empty_placeholder(['s2_scene_count']),
        )
    )

    elevation = NASA_srtm.log().divide(10).clamp(0, 1).rename('Elevation')
    l8_img = safe_landsat_median(l8_ic, L8_RENAME)
    l9_img = safe_landsat_median(l9_ic, L9_RENAME)
    modis_img = safe_median(modis_ic, ['Lai'], ['LAI'])
    smap_img = safe_median(smap_ic, ['sm_surface'], ['MOISTURE'])

    return s2_median.addBands([s2_count, elevation, l8_img, l9_img, modis_img, smap_img])


def prepare_fc_for_period(period, fields_fc):
    start_date_str = period['fortnight_start']
    end_date_str = (datetime.strptime(start_date_str, '%Y-%m-%d') + timedelta(days=14)).strftime('%Y-%m-%d')
    year = period['year']

    return fields_fc.map(lambda f: f.set({
        'year': year,
        'fortnight_start': start_date_str,
        'fortnight_end': end_date_str,
        'Area': f.get('area'),
        'state': f.get('state'),
    }))


def reduce_period(fields_fc, period, region=None):
    start_date_str = period['fortnight_start']
    end_date_str = (datetime.strptime(start_date_str, '%Y-%m-%d') + timedelta(days=14)).strftime('%Y-%m-%d')
    if region is None:
        region = EXTRACTION_REGION
    composite = build_period_composite(start_date_str, end_date_str, region)
    tagged_fc = prepare_fc_for_period(period, fields_fc)

    return composite.reduceRegions(
        collection=tagged_fc,
        reducer=ee.Reducer.mean(),
        scale=EXPORT_SCALE,
        tileScale=EXPORT_TILE_SCALE,
    )


def export_period_task(fields_fc, period, region=None):
    start_date_str = period['fortnight_start']
    state_tag = (STATE_FILTER or 'ALL').replace(' ', '')
    desc = f"fortnight_{state_tag}_{start_date_str.replace('-', '')}"
    reduced = reduce_period(fields_fc, period, region=region)

    task = ee.batch.Export.table.toDrive(
        collection=reduced,
        description=desc,
        folder=EXPORT_FOLDER,
        fileNamePrefix=desc,
        fileFormat='CSV',
    )
    task.start()
    return task, desc


def periods_by_starts(start_dates):
    starts = set(start_dates)
    return [p for p in fortnight_periods if p['fortnight_start'] in starts]


def resubmit_failed_exports(tasks, periods, fields_fc, failed_starts=None):
    """Re-submit exports for failed tasks or explicit fortnight start dates."""
    if failed_starts is None:
        failed_starts = [
            periods[i]['fortnight_start']
            for i, task in enumerate(tasks)
            if task.status().get('state') == 'FAILED'
        ]

    retry_periods = periods_by_starts(failed_starts)
    retry_tasks = []
    retry_descriptions = []

    print(f'Re-submitting {len(retry_periods)} failed export tasks...')
    for period in retry_periods:
        task, desc = export_period_task(fields_fc, period)
        retry_tasks.append(task)
        retry_descriptions.append(desc)
        print(f"  {desc}")

    return retry_tasks, retry_descriptions, failed_starts

In [ ]:
import time

export_tasks = []
export_descriptions = []

if EXTRACT_MODE == 'interactive':
    preview_period = next(
        p for p in fortnight_periods if p['fortnight_start'] == INTERACTIVE_PREVIEW_START
    )
    start = preview_period['fortnight_start']
    end = (datetime.strptime(start, '%Y-%m-%d') + timedelta(days=14)).strftime('%Y-%m-%d')
    print(f"Interactive preview: {start} → {end}")

    preview_fc = fc.limit(MAX_FIELDS or 10)
    reduced = reduce_period(preview_fc, preview_period)
    feature_data = [f['properties'] for f in reduced.getInfo()['features']]
    print(f"Preview rows: {len(feature_data)}")
    print('Sample columns:', sorted(feature_data[0].keys())[:12], '...')

else:
    print(f"Submitting {len(fortnight_periods)} batch export tasks to Drive folder '{EXPORT_FOLDER}'...")
    for i, period in enumerate(fortnight_periods, start=1):
        task, desc = export_period_task(fc, period)
        export_tasks.append(task)
        export_descriptions.append(desc)
        if i % 10 == 0 or i == len(fortnight_periods):
            print(f"  submitted {i}/{len(fortnight_periods)}: {desc}")

    print('All tasks submitted. Monitor in Colab or at https://code.earthengine.google.com/tasks')

Submitting 72 batch export tasks to Drive folder 'GEE_Fortnight_Exports'...
  submitted 10/72: fortnight_Texas_20190515
  submitted 20/72: fortnight_Texas_20191015
  submitted 30/72: fortnight_Texas_20210315
  submitted 40/72: fortnight_Texas_20210815
  submitted 50/72: fortnight_Texas_20220115
  submitted 60/72: fortnight_Texas_20220615
  submitted 70/72: fortnight_Texas_20221115
  submitted 72/72: fortnight_Texas_20221215
All tasks submitted. Monitor in Colab or at https://code.earthengine.google.com/tasks


In [ ]:
# Re-submit failed exports only (run after polling shows failures)
# Option A: auto-detect from prior export_tasks list
# retry_tasks, retry_descriptions, failed_starts = resubmit_failed_exports(export_tasks, fortnight_periods, fc)

# Option B: explicit failed fortnight starts from your last run
FAILED_PERIOD_STARTS = [
    '2019-01-01', '2019-01-15', '2019-02-01', '2019-02-15', '2019-03-01', '2019-03-15',
    '2019-04-01', '2019-04-15', '2019-05-01', '2019-05-15', '2019-06-01', '2019-06-15',
    '2019-07-01', '2019-07-15', '2019-08-01', '2019-08-15', '2019-09-01', '2019-09-15',
    '2019-10-01', '2019-10-15', '2019-11-01', '2019-11-15', '2019-12-01', '2019-12-15',
    '2021-01-01', '2021-01-15', '2021-02-01', '2021-02-15', '2021-03-01', '2021-03-15',
    '2021-04-01', '2021-04-15', '2021-05-01', '2021-05-15', '2021-06-01', '2021-06-15',
    '2021-07-01', '2021-07-15', '2021-08-01', '2021-08-15', '2021-09-01', '2021-09-15',
    '2021-10-01', '2021-10-15', '2022-04-01',
]

RETRY_FAILED = False  # set True to re-submit FAILED_PERIOD_STARTS

if RETRY_FAILED:
    retry_tasks, retry_descriptions, failed_starts = resubmit_failed_exports(
        export_tasks, fortnight_periods, fc, failed_starts=FAILED_PERIOD_STARTS
    )
    export_tasks = retry_tasks
    export_descriptions = retry_descriptions
    print(f'Re-submitted {len(retry_tasks)} tasks')
else:
    print('Set RETRY_FAILED = True to re-submit failed fortnights')

Set RETRY_FAILED = True to re-submit failed fortnights


In [ ]:

def poll_export_tasks(tasks, interval_sec=30):
    """Poll Earth Engine export tasks until all finish."""
    if not tasks:
        print('No export tasks to poll.')
        return

    while True:
        statuses = [t.status().get('state') for t in tasks]
        completed = statuses.count('COMPLETED')
        failed = statuses.count('FAILED')
        running = len(statuses) - completed - failed - statuses.count('CANCELLED')
        print(f"Tasks — completed: {completed}, running: {running}, failed: {failed}, total: {len(tasks)}")

        if all(s in ('COMPLETED', 'FAILED', 'CANCELLED') for s in statuses):
            if failed:
                for t in tasks:
                    st = t.status()
                    if st.get('state') == 'FAILED':
                        print('FAILED:', st.get('description'), st.get('error_message'))
            break
        time.sleep(interval_sec)


if EXTRACT_MODE == 'export' and export_tasks:
    poll_export_tasks(export_tasks)
elif EXTRACT_MODE == 'interactive':
    df = pd.DataFrame(feature_data)
    if 'state' in df.columns and 'year' in df.columns:
        df['state_annual_yield'] = df.apply(
            lambda r: state_yield_lookup.get((r.get('state'), r.get('year'))), axis=1
        )
    print(df.shape)
    df.head()

Tasks — completed: 5, running: 67, failed: 0, total: 72


Tasks — completed: 12, running: 60, failed: 0, total: 72
Tasks — completed: 19, running: 53, failed: 0, total: 72
Tasks — completed: 25, running: 47, failed: 0, total: 72
Tasks — completed: 33, running: 39, failed: 0, total: 72
Tasks — completed: 38, running: 34, failed: 0, total: 72
Tasks — completed: 51, running: 21, failed: 0, total: 72
Tasks — completed: 64, running: 8, failed: 0, total: 72
Tasks — completed: 72, running: 0, failed: 0, total: 72


In [ ]:
import glob
import os


def find_export_csvs():
    """Locate fortnight export CSVs — EE may write outside the expected Drive subfolder."""
    state_tag = (STATE_FILTER or 'ALL').replace(' ', '')
    filename_pattern = f'fortnight_{state_tag}_*.csv'

    search_roots = [
        DRIVE_EXPORT_PATH,
        f'/content/drive/MyDrive/{EXPORT_FOLDER}',
        '/content/drive/MyDrive',
        '/content/drive/MyDrive/Google Earth Engine',
    ]

    found = set()
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        found.update(glob.glob(os.path.join(root, filename_pattern)))
        found.update(glob.glob(os.path.join(root, '**', filename_pattern), recursive=True))

    return sorted(found)


if EXTRACT_MODE == 'export':
    csv_files = []
    for attempt in range(12):
        csv_files = find_export_csvs()
        if csv_files:
            break
        print(f'Waiting for Drive sync... attempt {attempt + 1}/12 (tasks done ≠ files visible yet)')
        time.sleep(30)

    if not csv_files:
        print('Searched paths:')
        for root in [DRIVE_EXPORT_PATH, f'/content/drive/MyDrive/{EXPORT_FOLDER}', '/content/drive/MyDrive']:
            print(' ', root, '→', os.path.isdir(root))
        raise FileNotFoundError(
            f"No CSV files matching fortnight exports found on Drive. "
            f"Check Google Drive for folder '{EXPORT_FOLDER}' or files like fortnight_Texas_20190101.csv, "
            f"then set DRIVE_EXPORT_PATH to the correct folder and re-run this cell."
        )

    print(f'Found {len(csv_files)} export files')
    print('First file:', csv_files[0])

    df = pd.concat([pd.read_csv(path) for path in csv_files], ignore_index=True)

    if 'state' in df.columns and 'year' in df.columns:
        df['state_annual_yield'] = df.apply(
            lambda r: state_yield_lookup.get((r.get('state'), int(r.get('year')))), axis=1
        )

    print(f'Merged {len(csv_files)} export files → {df.shape[0]} rows')
    df.head()

df.to_csv(OUTPUT_CSV, index=False)
print('Saved:', OUTPUT_CSV)

if IN_COLAB and EXTRACT_MODE == 'interactive':
    from google.colab import files
    files.download(OUTPUT_CSV)

Found 72 export files
First file: /content/drive/MyDrive/GEE_Fortnight_Exports (1)/fortnight_Texas_20190101.csv
Merged 72 export files → 50832 rows
Saved: usa_fortnight_features.csv



## Feature weights (Phase 3 reference)

Used later in yield mapping — not applied during extraction.


In [ ]:

feature_weights = {
    'field_id': 0,
    'state': 0,
    'fortnight_start': 0,
    'fortnight_end': 0,
    'NDVI': 0.15,
    'EVI': 0.10,
    'GNDVI': 0.05,
    'SAVI': 0.05,
    'B1': 0.03, 'B2': 0.03, 'B3': 0.03, 'B4': 0.03,
    'B5': 0.03, 'B6': 0.03, 'B7': 0.03, 'B8': 0.03,
    'B9': 0.03, 'B11': 0.03, 'B12': 0.03,
    'SR8_B1': 0.03, 'SR8_B2': 0.03, 'SR8_B3': 0.03, 'SR8_B4': 0.03,
    'SR8_B5': 0.03, 'SR8_B6': 0.03, 'SR8_B7': 0.03,
    'NBR_8': 0.02, 'NDMI_8': 0.02, 'NDWI_8': 0.02,
    'NDBI_8': 0.02, 'NDBaI_8': 0.02, 'MNDWI_8': 0.02,
    'SR9_B1': 0.02, 'SR9_B2': 0.02, 'SR9_B3': 0.02, 'SR9_B4': 0.02,
    'SR9_B5': 0.02, 'SR9_B6': 0.02, 'SR9_B7': 0.02,
    'NBR_9': 0.02, 'NDMI_9': 0.02, 'NDWI_9': 0.02,
    'NDBI_9': 0.02, 'NDBaI_9': 0.02, 'MNDWI_9': 0.02,
    'LAI': 0.05,
    'MOISTURE': 0.05,
    'Area': 0.20,
    'Elevation': 0.10,
    'VCI': 0.10, 'TVI': 0.10,
    'BI': 0.05, 'BI2': 0.05,
    'CI': 0.05, 'CI1': 0.05,
    'SATVI': 0.10, 'HVSI': 0.10,
    'SOCI': 0.05, 'ASI': 0.05,
    'BSI': 0.05, 'MSAVI': 0.10,
}
sum(feature_weights.values())

2.52